## 1 - Configurações Gerais

In [1]:
#@markdown Detectando o Ambiente de Execução { display-mode: "form" }
try:
    import google.colab
    EXECUTION_ENV = "colab"
except ImportError:
    try:
        import kaggle_secrets
        EXECUTION_ENV = "kaggle"
    except ImportError:
        EXECUTION_ENV = "local"

print(f"AMBIENTE DE EXECUÇÃO DETECTADO: {EXECUTION_ENV}")

AMBIENTE DE EXECUÇÃO DETECTADO: local


In [2]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    !pip install -q minigrid==3.1.0
    !pip install -q langchain_openai

In [3]:
import os
import sys

if EXECUTION_ENV == "colab":
    repository_path = os.path.join(os.getcwd(), "minigrid_benchmark")

if EXECUTION_ENV == "kaggle":
    repository_path = os.path.join("/kaggle/working/", "minigrid_benchmark")

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(repository_path):
        !git clone https://github.com/pablo-sampaio/minigrid_benchmark.git
    repository_src_path = os.path.join(repository_path, "src")
    sys.path.append(repository_src_path)
    print(f"Repository src path: {repository_src_path}")

In [4]:
from langchain_openai import ChatOpenAI

import experiments_util
from experiments_util import run_and_save_experiments, create_experiment_config

In [5]:
if EXECUTION_ENV == "colab":
    # Mount Google Drive, to save results there
    from google.colab import drive
    drive.mount('/content/drive')
    results_dir = '/content/drive/My Drive/EAD-Pesquisa-Agentes/_projeto_minigrid/results'

if EXECUTION_ENV == "kaggle":
    results_dir = "/kaggle/working/results"

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    experiments_util.RESULTS_BASE_DIR = results_dir

In [6]:
OPENAI_KEY_NAME = "OPENAI_PABLO"
OPENAI_API_KEY = None

if EXECUTION_ENV == "colab":
    # just assert that HUGGINGFACE_API_KEY or KF_TOKEN secret is set
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get(OPENAI_KEY_NAME)

if EXECUTION_ENV == "kaggle":
    # para o Kaggle
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    OPENAI_API_KEY = user_secrets.get_secret(OPENAI_KEY_NAME)

if EXECUTION_ENV == "local":
    OPENAI_API_KEY = os.getenv(OPENAI_KEY_NAME)

if not OPENAI_API_KEY:
    raise ValueError(f"{OPENAI_KEY_NAME} environment variable is not set")

## 2 - Configurar o Modelo

In [7]:
MODEL_GPT_5_4 = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key=OPENAI_API_KEY,
)

MODEL_GPT_4_1 = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=OPENAI_API_KEY,
)

## 3 - Configurações do Experimento

In [8]:
configs = [
    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, True, False, False),
    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, True, True, True),
    create_experiment_config("gpt-4.1", MODEL_GPT_4_1, True, False, False),
    create_experiment_config("gpt-4.1", MODEL_GPT_4_1, True, True, True),

    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, True, False, False, history_size=5),
    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, True, True, True, history_size=5),
    create_experiment_config("gpt-4.1", MODEL_GPT_4_1, True, False, False, history_size=5),
    create_experiment_config("gpt-4.1", MODEL_GPT_4_1, True, True, True, history_size=5),

    # experimento de historico (para comparar 1, 3, 5 e 9)
    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, True, False, False, history_size=3),
    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, True, True, True, history_size=3),
    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, True, False, False, history_size=9),
    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, True, True, True, history_size=9),

    # experimentos com visao local
    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, False, False, True, history_size=1),
    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, False, False, True, history_size=3),
    create_experiment_config("gpt-5.4", MODEL_GPT_5_4, False, False, True, history_size=5),
]

## 4 - Execução do Experimento

In [9]:
experiment_name = "2026-05-30_openai"  # se já existir, continua de onde parou

In [10]:
##########################
##  Execute experiments ##
##########################

final_results, filepath = run_and_save_experiments(configs, experiment_name=experiment_name, verbose=True)


Results will be saved to: c:\Users\pablo\Documents\Projetos\pablo-sampaio\minigrid_benchmark\results\2026-05-30_openai\2026-05-30_openai.json


Completed Experiment Configurations:   0%|          | 0/15 [00:00<?, ?it/s]

Completed Environments:   0%|          | 0/2 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Run 1 failed due to API error. Retrying episode in 30s (attempt 1/2). Last error: Failed to get response from model after 3 attempts. Last error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************9UUA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


RuntimeError: Failed to get response from model after 3 attempts. Last error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************9UUA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    # zip the final results directory
    import shutil

    experiment_result_dir = os.path.dirname(filepath)
    zip_path = os.path.join(experiments_util.RESULTS_BASE_DIR, f"{experiment_name}_results_zip")

    # Creates results_zip.zip containing everything
    shutil.make_archive(zip_path, 'zip', experiment_result_dir)

    print(f"Created: {zip_path}.zip")

In [ ]:
print("Saved results to:", filepath)
print("Now, run notebook \"run_experiments_analysis.ipynb\".")

Now, run notebook "run_experiments_analysis.ipynb".
